In [1]:
import numpy as np
import sounddevice as sd
import threading

In [2]:
from dissipative_sonification import config as cfg

In [3]:
v_row = np.array([20, 4, 60, 17, 8, 2, 4, 5, 56, 34, 123, 12], dtype=np.float32) / 100
v_row

array([0.2 , 0.04, 0.6 , 0.17, 0.08, 0.02, 0.04, 0.05, 0.56, 0.34, 1.23,
       0.12], dtype=float32)

In [4]:
idx = np.random.randint(0, len(v_row), cfg.MAX_VOICES)
idx

array([ 1, 10,  0,  6,  5,  6,  4,  6])

In [5]:
v_row[idx]

array([0.04, 1.23, 0.2 , 0.04, 0.02, 0.04, 0.08, 0.04], dtype=float32)

In [6]:
pitches = v_row[idx] * cfg.MAX_VOICES


In [7]:
pitches

array([0.32, 9.84, 1.6 , 0.32, 0.16, 0.32, 0.64, 0.32], dtype=float32)

In [8]:
pitches.astype(int)

array([0, 9, 1, 0, 0, 0, 0, 0])

In [9]:
pitches_cliped = np.clip(pitches.astype(int), 0, cfg.MAX_VOICES - 1)

In [10]:
pitches_cliped

array([0, 7, 1, 0, 0, 0, 0, 0])

In [11]:
freqs = cfg.BASE_FREQ * (2 ** (cfg.SCALE_DEGREES[pitches_cliped] / 12.0))

In [12]:
freqs

array([440.        , 880.        , 493.88330126, 440.        ,
       440.        , 440.        , 440.        , 440.        ])

In [13]:
v_mean = np.mean(v_row)

In [14]:
v_mean

np.float32(0.2875)

In [15]:
amps = np.ones(cfg.MAX_VOICES) * v_mean

In [16]:
amps

array([0.28749999, 0.28749999, 0.28749999, 0.28749999, 0.28749999,
       0.28749999, 0.28749999, 0.28749999])

In [17]:
starting_phases = np.zeros(cfg.MAX_VOICES)

In [18]:
starting_phases

array([0., 0., 0., 0., 0., 0., 0., 0.])

In [19]:
# Shared state
'''
global_parameters = {"freq": 440.0, "amp": 0.5}
lock = threading.Lock()
'''

'\nglobal_parameters = {"freq": 440.0, "amp": 0.5}\nlock = threading.Lock()\n'

In [20]:
# phase = 0 # starting phase for the current block

In [26]:
def callback(outdata, frames, time, status):

    global freqs, amps, starting_phases

    # print(phase)
    '''
    with lock:
        freq = global_parameters["freq"]
        amp = global_parameters["amp"]
    '''
    voices = []
    for v in range(cfg.MAX_VOICES): # simultaneously press some notes of Dorian scale
        # freq is measured in cycles per second
        radians_per_second = 2 * np.pi * freqs[v] # radians / second
        sample_duration = 1 / cfg.SAMPLE_RATE # seconds
        phase_increment_per_sample = radians_per_second * sample_duration # radians
        angles = (
            phase_increment_per_sample * np.arange(frames)
        ) # appropriate phase advance for each frame (for each sample) in the block
        phases = starting_phases[v] + angles
        starting_phases[v] = (starting_phases[v] + phase_increment_per_sample * frames) % (2 * np.pi)    
        voices.append(amps[v] * np.sin(phases))
    
    outdata[:, 0] = sum(voices)
    outdata[:, 1] = sum(voices)

    

In [27]:
# Start stream
stream = sd.OutputStream(
            samplerate=cfg.SAMPLE_RATE,
            channels=2,
            dtype="float32",
            blocksize=0,
            callback=callback,
        )
stream.start()

In [28]:
# In another cell, update parameters safely:
'''
with lock:
    global_parameters.update({"freq": 880.0, "amp": 0.3})
'''

'\nwith lock:\n    global_parameters.update({"freq": 880.0, "amp": 0.3})\n'

In [29]:
stream.stop()

In [30]:
stream.close()

In [2]:
import dissipative_sonification.config as cfg
from dissipative_sonification.sonification import AudioEngine

In [3]:
engine = AudioEngine()

In [4]:
engine.start()

In [8]:
state = {
            "f": 0.3,
            "k": 680.0,
            "v_mean": 0.0,
            "v_row": np.zeros(cfg.WIDTH),
            "entropy": 0.0,
            "gradient": 0.0,
            "pan": 0.0,
        }
engine.update_state(state)

In [9]:
engine.stop()